# Evaluate Term Dispersion Scores on the BC5CDR Corpus Data and Reproduce Results Reported in the Manuscript 

Description: Evaluate the following term dispersion score/keyword extraction methods on the BC5CDR corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)
- KeyBERT
- KeyLLM

Calculate average P@k scores for each scoring function using the BC5CDR terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords.

This version of the code includes singletons in the analysis.

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the BC5CDR Corpus Data

In particular, we load the preprocessed BC5CDR corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., Chemical and Disease).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed BC5CDR corpus documents
bc5cdr_corpus_path = '../../1-preprocessing/bc5cdr-preprocessed.json'

with open(bc5cdr_corpus_path, "r") as j:
  bc5cdr_corpus = json.loads(j.read())

# Load gold standard terms 
bc5cdr_keywords_path = '../../1-preprocessing/bc5cdr-keywords.tsv'

with open(bc5cdr_keywords_path, "r") as c:
  bc5cdr_lexes_and_sems = pd.read_csv(c, sep='\t')

bc5cdr_lexes = bc5cdr_lexes_and_sems.lex.to_numpy()
bc5cdr_lexes_and_sems['sem'] = bc5cdr_lexes_and_sems['sem'].str.lower()
bc5cdr_semantic_class_names = bc5cdr_lexes_and_sems['sem'].unique().tolist()

# Print to console
display(bc5cdr_lexes_and_sems)

,lex,sem
0,naloxone_lex,chemical
1,clonidine_lex,chemical
2,hypertensive_lex,disease
3,nalozone_lex,chemical
4,hypotensive_lex,disease
...,...,...
4866,galactose_lex,chemical
4867,dgalactose_lex,chemical
4868,dglucose_lex,chemical
4869,memory_deterioration_lex,disease


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [3]:
# Compile the BC5CDR corpus vocabulary
pre_vocab = []
for i in range(len(bc5cdr_corpus)):
  pre_vocab.append(bc5cdr_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert bc5cdr documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(bc5cdr_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [4]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
B_i.A[0][11764] = 1459 # Hack to fix idiosyncracy with brentq solver for stopword 'of'
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [5]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/bc5cdr/2-tables/singletons-included-analysis/../../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [6]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, bc5cdr_lexes_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF


Term dispersion scores:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,1.098241


In [7]:
# Integrate KeyBERT scores

# Load KeyBERT results 
keybert_keywords_path = '../keybert-scores.tsv'

with open(keybert_keywords_path, "r") as c:
  keybert_scores_df = pd.read_csv(c, sep='\t')

# Replace blanks by NANs in sems column
keybert_scores_df['sem'] = keybert_scores_df['sem'].fillna(str())

# Convert semantic classes to lowercase
keybert_scores_df['sem'] = keybert_scores_df['sem'].str.lower()

# Add KeyBERT scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keybert_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyBERT'] = term_scores_aug_df['KeyBERT'].fillna(0)

# Check for duplicate terms
all_duplicates = keybert_scores_df[keybert_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keybert_scores_df = keybert_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyBERT


Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,0.000000,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,0.000000,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,0.000000,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,0.000667,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,0.000000,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,1.098241


In [8]:
# Integrate KeyLLM scores

# Load KeyLLM results 
keyllm_keywords_path = '../keyllm-scores.tsv'

with open(keyllm_keywords_path, "r") as c:
  keyllm_scores_df = pd.read_csv(c, sep='\t')

# Remove rows with empty term entries
keyllm_scores_df = keyllm_scores_df.dropna(subset=['term'])

# Replace blanks by NANs in sems column
keyllm_scores_df['sem'] = keyllm_scores_df['sem'].fillna(str())

# Convert semantic classes to lowercase
keyllm_scores_df['sem'] = keyllm_scores_df['sem'].str.lower()

# Check for duplicate terms
all_duplicates = keyllm_scores_df[keyllm_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keyllm_scores_df = keyllm_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Print to console
print("KeyLLM scores:")
display(keyllm_scores_df)

# Add KeyLLM scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keyllm_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyLLM'] = term_scores_aug_df['KeyLLM'].fillna(0)

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'KeyLLM', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyLLM


KeyLLM scores:


,term,sem,KeyLLM
1,112dihydro2acenaphthylenylpiperazine_lex,chemical,0.000667
2,11deoxycortisol_lex,chemical,0.000667
3,11dichloro222trifluoroethane_lex,chemical,0.000667
4,11ketopregnenolone_sulphate_lex,chemical,0.000667
5,125dihydroxyvitamin_d_lex,chemical,0.000667
...,...,...,...
9101,methicillin_lex,chemical,0.000000
9102,tazobactam_lex,chemical,0.000000
9103,galactose_lex,chemical,0.000000
9104,dgalactose_lex,chemical,0.000000


Duplicate rows:


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF


,term,sem,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,0001abstract,NaN,1,1,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,0.000000,0.000000,-0.000371
1,0014unit,NaN,1,3,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,0.000000,1.098241
2,001abstract,NaN,3,3,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,0.000000,-0.001195
3,0070unit,NaN,1,1,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,0.000000,0.000000,-0.000371
4,0075mgkg,NaN,1,1,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,0.000000,0.000000,-0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,chemical,1,1,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,0.000667,0.000667,-0.000371
17951,zung,NaN,1,1,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,0.000000,0.000000,-0.000371
17952,zungconde,NaN,1,2,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.000000,0.692776
17953,zyban_lex,chemical,1,3,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,0.000667,1.098241


In [9]:
# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t', index=False)

## Compile BC5CDR Corpus Summary Statistics

This is the result of Table 4 from the manuscript.

In [10]:
# Count number of distinct lexical units in each semantic class
lex_counts = term_scores_aug_df.dropna(subset=['sem']).groupby('sem')['term'].nunique().reindex(bc5cdr_semantic_class_names).to_list()

# Count number of annotations associated with each semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['sem']).groupby('sem')['ni'].sum().reindex(bc5cdr_semantic_class_names).to_list()

# Count number of singletons associated with each semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['sem']).groupby('sem')['ni'].sum().reindex(bc5cdr_semantic_class_names).to_list()

# Initialize BC5CDR summary statistics data frame
bc5cdr_summary_stats_df = pd.DataFrame({
    'Semantic class': bc5cdr_semantic_class_names,
    'Unique terms': lex_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print BC5CDR summary statistics to console
display(bc5cdr_summary_stats_df)

,Semantic class,Unique terms,Annotations,Singletons
0,chemical,2086,15819,720
1,disease,2757,12630,1347


## Terminology Extraction Task Experiment

Here we reproduce the result of Tables 5, 6, 7, A1, A2, and A3 from the manuscript.

In [12]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'ni', 'bi'])

# Print to console
display(term_scores_df)

# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-minimal.tsv', sep='\t', index=False)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,0001abstract,7.313220,12.557595,0.702910,1.0,278.000000,-0.000978,0.000000,0.000000,-0.000371
1,0014unit,7.313220,11.458983,680.512989,3.0,480.000000,-0.000563,0.000000,0.000000,1.098241
2,001abstract,6.214608,11.458983,0.673894,1.0,302.666667,-0.003194,0.000000,0.000000,-0.001195
3,0070unit,7.313220,12.557595,0.702910,1.0,160.000000,-0.000563,0.000000,0.000000,-0.000371
4,0075mgkg,7.313220,12.557595,0.702910,1.0,100.000000,-0.000352,0.000000,0.000000,-0.000371
...,...,...,...,...,...,...,...,...,...,...
17950,zuclopenthixol_lex,7.313220,12.557595,0.702910,1.0,131.000000,-0.000461,0.000667,0.000667,-0.000371
17951,zung,7.313220,12.557595,0.702910,1.0,200.000000,-0.000704,0.000000,0.000000,-0.000371
17952,zungconde,7.313220,11.864448,234.217595,2.0,260.000000,-0.000457,0.000000,0.000000,0.692776
17953,zyban_lex,7.313220,11.458983,680.512989,3.0,447.000000,-0.000524,0.000667,0.000667,1.098241


Define various functions used in the analysis.

In [13]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [14]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(179247)

# Set number of replicates
R = 100 # To test, set to 5

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
all_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'].isin(['chemical', 'disease']), 'lex'].tolist()
chemical_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'] == 'chemical', 'lex'].tolist()
disease_lexes = bc5cdr_lexes_and_sems.loc[bc5cdr_lexes_and_sems['sem'] == 'disease', 'lex'].tolist()

categories = {
    'all': all_lexes,
    'chemical': chemical_lexes,
    'disease': disease_lexes}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                                                                | 0/100 [00:00<?, ?it/s]

r = 0


  1%|█▎                                                                                                                                      | 1/100 [00:13<22:36, 13.70s/it]

r = 1


  2%|██▋                                                                                                                                     | 2/100 [00:27<22:26, 13.74s/it]

...

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 98/100 [22:36<00:27, 13.84s/it]

r = 98


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 99/100 [22:50<00:13, 13.90s/it]

r = 99


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [23:03<00:00, 13.84s/it]


Save evaluation metrics as Pkl files.

In [15]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [16]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-a8', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-a8/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-a8/chemical-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-a8/disease-fk-means.csv', index=False)

In [17]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0013,0.0011,0.0023,0.0027,0.0029,0.0020,0.0041,0.0041,0.0027
50,0.0067,0.0056,0.0115,0.0134,0.0114,0.0108,0.0203,0.0203,0.0134
100,0.0132,0.0111,0.0227,0.0299,0.0262,0.0237,0.0402,0.0402,0.0299
500,0.0608,0.0527,0.1053,0.1254,0.1128,0.0998,0.1862,0.1862,0.1262
1000,0.1118,0.0964,0.1936,0.2141,0.1904,0.1712,0.3407,0.3407,0.2121
5000,0.3325,0.2885,0.4260,0.4318,0.3706,0.3924,0.8991,0.8033,0.4354


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0012,0.0009,0.0037,0.0054,0.0047,0.0000,0.0028,0.0019,0.0054
50,0.0063,0.0043,0.0183,0.0256,0.0203,0.0102,0.0199,0.0198,0.0256
100,0.0124,0.0085,0.0360,0.0583,0.0461,0.0186,0.0380,0.0390,0.0582
500,0.0528,0.0378,0.1533,0.2155,0.1875,0.0681,0.1739,0.1730,0.2158
1000,0.0891,0.0629,0.2595,0.3131,0.2737,0.1118,0.2839,0.2868,0.3068
5000,0.1959,0.1392,0.3363,0.3403,0.3000,0.2159,0.5171,0.4755,0.3430


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0014,0.0013,0.0011,0.0007,0.0014,0.0036,0.0051,0.0058,0.0007
50,0.0068,0.0065,0.0060,0.0038,0.0043,0.0111,0.0203,0.0204,0.0038
100,0.0134,0.0127,0.0116,0.0069,0.0098,0.0269,0.0406,0.0398,0.0070
500,0.0580,0.0565,0.0506,0.0339,0.0356,0.1099,0.1675,0.1682,0.0349
1000,0.1009,0.0984,0.0875,0.0751,0.0708,0.1748,0.2970,0.2946,0.0772
5000,0.2435,0.2394,0.2337,0.2374,0.1964,0.3013,0.6699,0.5861,0.2395
